# CampaignLift: Uplift Modeling & Candidate Selection on Kaggle

This notebook orchestrates the model training, validation, candidate selection, and test scoring workflow for CampaignLift.

### Strict Architecture & Anti-Leakage Contracts
1. **Input Isolation**: Training reads strictly from `/kaggle/input/campaignlift-dataset/` (containing `features.json`, `splits.json`, and `manifest.json`).
2. **Oracle Separation**: The counterfactual synthetic oracle (`hidden_uplift.json`) is **NEVER** an input to training. If attached for post-selection evaluation, it must reside in an isolated dataset `/kaggle/input/campaignlift-oracle/`.
3. **Execution Script Contract**: This notebook sets path parameters and invokes the canonical training runner (`campaignlift_ml.train`). It does not invent ad-hoc model logic or hardcode metrics.
4. **No Live Inference**: Kaggle is used strictly for model fitting and artifact production. The resulting artifact package is downloaded for FastAPI serving.

In [ ]:
import os
import sys
from pathlib import Path

# -----------------------------------------------------------------------------
# 1. Path Contract & Environment Resolution
# -----------------------------------------------------------------------------
IN_KAGGLE = os.path.exists("/kaggle")

if IN_KAGGLE:
    # Standard Kaggle input/working paths
    DATASET_DIR = Path("/kaggle/input/campaignlift-dataset")
    OUTPUT_DIR = Path("/kaggle/working/outputs")
    ORACLE_DIR = Path("/kaggle/input/campaignlift-oracle")
    CODE_ROOT = Path("/kaggle/working/ml/src")
else:
    # Local workstation fallback (smoke / dev testing)
    REPO_ROOT = Path("..").resolve().parent
    DATASET_DIR = REPO_ROOT / "data" / "fixtures" / "fixture_v1"
    OUTPUT_DIR = REPO_ROOT / "ml" / "outputs"
    ORACLE_DIR = None
    CODE_ROOT = REPO_ROOT / "ml" / "src"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if str(CODE_ROOT) not in sys.path:
    sys.path.insert(0, str(CODE_ROOT))

print(f"[Environment] IN_KAGGLE: {IN_KAGGLE}")
print(f"[Path Contract] DATASET_DIR: {DATASET_DIR}")
print(f"[Path Contract] OUTPUT_DIR:  {OUTPUT_DIR}")
print(f"[Path Contract] ORACLE_DIR:  {ORACLE_DIR if (ORACLE_DIR and ORACLE_DIR.exists()) else 'Not attached (training pure)'}")
print(f"[Path Contract] CODE_ROOT:   {CODE_ROOT}")

In [ ]:
# -----------------------------------------------------------------------------
# 2. Dependency Verification & Integrity Checks
# -----------------------------------------------------------------------------
import numpy as np
import pandas as pd
import sklearn
import lightgbm as lgb
import joblib
import yaml

print(f"[Dependency Check] NumPy:        {np.__version__}")
print(f"[Dependency Check] Pandas:       {pd.__version__}")
print(f"[Dependency Check] Scikit-Learn: {sklearn.__version__}")
print(f"[Dependency Check] LightGBM:     {lgb.__version__}")
print(f"[Dependency Check] Joblib:       {joblib.__version__}")
print(f"[Dependency Check] PyYAML:       {yaml.__version__}")

# Verify dataset directory contains required training files
required_files = ["features.json", "splits.json", "manifest.json"]
missing_files = [f for f in required_files if not (DATASET_DIR / f).exists()]
if missing_files:
    raise FileNotFoundError(f"Dataset directory '{DATASET_DIR}' is missing required files: {missing_files}")
print(f"[Dataset Check] All required dataset files present in '{DATASET_DIR}'.")

# Verify hidden_uplift is NOT present inside the training dataset directory
forbidden_leak = DATASET_DIR / "hidden_uplift.json"
if forbidden_leak.exists():
    raise RuntimeError(
        "CRITICAL CAUSAL LEAKAGE: 'hidden_uplift.json' was found inside the training dataset directory! "
        "Oracle potential outcomes must NEVER be colocated with feature tables."
    )
print("[Anti-Leakage Check] Verified: 'hidden_uplift.json' is NOT present in training inputs.")

### 3. Execution of Canonical Training Script

The training pipeline executes the 10 stages defined in `planning/kaggle_plan.md`:
1. Print dataset version, seed, and file checksums from manifest.
2. Assert forbidden columns are absent against schema denylist.
3. Load train and validation splits only.
4. Fit all candidates: B0 (baseline), B1 (logistic S-learner), U0 (logistic T-learner), U1 (LightGBM T-learner), U2 (LightGBM S-learner).
5. Compute validation metrics and write `outputs/validation_metrics.json`.
6. Apply the validation Qini selection rule and write `outputs/selection.json`.
7. Refit the selected winner on train + validation.
8. Touch test split once and write `outputs/test_metrics.json`.
9. (Optional) Run oracle evaluation if `-oracle` dataset is attached, writing `outputs/oracle_metrics.json`.
10. Package model artifact and `metadata.json` for download.

In [ ]:
# -----------------------------------------------------------------------------
# 3. Run Training Entry Point
# -----------------------------------------------------------------------------
import subprocess

train_cmd = [
    sys.executable,
    "-m",
    "campaignlift_ml.train",
    "--dataset-dir", str(DATASET_DIR),
    "--output-dir", str(OUTPUT_DIR),
]

if ORACLE_DIR and ORACLE_DIR.exists():
    train_cmd.extend(["--oracle-dir", str(ORACLE_DIR)])

print(f"[Execution] Invoking training runner: {' '.join(train_cmd)}")

# The training runner will be implemented in Step 12-14
# When run in Step 11 stub mode, verify CLI contract
try:
    result = subprocess.run(train_cmd, capture_output=True, text=True, check=True)
    print(result.stdout)
except subprocess.CalledProcessError as err:
    print(f"[Notice] Runner stub or execution returned: {err.returncode}")
    if err.stdout:
        print("STDOUT:", err.stdout)
    if err.stderr:
        print("STDERR:", err.stderr)
except Exception as e:
    print(f"[Notice] Runner entry point stub check: {e}")

In [ ]:
# -----------------------------------------------------------------------------
# 4. Output Artifact Verification
# -----------------------------------------------------------------------------
expected_outputs = [
    "validation_metrics.json",
    "selection.json",
    "test_metrics.json",
    "metadata.json",
]

print(f"[Outputs] Checking artifacts in '{OUTPUT_DIR}':")
for f in expected_outputs:
    path = OUTPUT_DIR / f
    status = "FOUND" if path.exists() else "PENDING RUN"
    print(f"  - {f:25s}: {status}")